In [1]:
import pandas as pd
import os
from pathlib import Path
from collections import defaultdict

In [5]:
import pandas as pd
import os
from pathlib import Path
from collections import defaultdict

# Set the images path
images_path = Path('datasets/images')

# Check if hierarchical or flat structure
is_hierarchical = any(p.is_dir() for p in images_path.iterdir())

data = []

if is_hierarchical:
    print("Detected hierarchical structure: category/label/image.jpg\n")
    
    # Hierarchical structure: images/category/label/image.jpg
    for category_dir in images_path.iterdir():
        if not category_dir.is_dir():
            continue
        
        category = category_dir.name
        
        for label_dir in category_dir.iterdir():
            if not label_dir.is_dir():
                continue
            
            label = label_dir.name
            
            # Get all images in this label directory
            image_files = list(label_dir.glob('*.jpg')) + \
                         list(label_dir.glob('*.png')) + \
                         list(label_dir.glob('*.tif'))
            
            for img_file in image_files:
                data.append({
                    'filename': img_file.name,
                    'filepath': str(img_file.relative_to(images_path)),
                    'label': label,
                    'category': category
                })
else:
    print("Detected flat structure: label_category000000.jpg\n")
    
    # Flat structure: images/label_category000000.jpg
    image_files = list(images_path.glob('*.jpg')) + \
                  list(images_path.glob('*.png')) + \
                  list(images_path.glob('*.tif'))
    
    for img_file in image_files:
        filename = img_file.stem
        
        parts = filename.split('_')
        if len(parts) >= 2:
            label = parts[0]
            category_info = parts[1]
            category = ''.join([c for c in category_info if not c.isdigit()])
            
            data.append({
                'filename': img_file.name,
                'filepath': img_file.name,
                'label': label,
                'category': category
            })

df = pd.DataFrame(data)

# Display EDA
print("=== Summary by Label ===")
print(df['label'].value_counts())
print("\n=== Summary by Category ===")
print(df['category'].value_counts())
print("\n=== Summary by Label and Category ===")
print(df.groupby(['label', 'category']).size().unstack(fill_value=0))
print(f"\n=== Total Images: {len(df)} ===")

# Additional statistics
print("\n=== Distribution Percentages ===")
print("\nBy Label:")
print((df['label'].value_counts() / len(df) * 100).round(2))
print("\nBy Category:")
print((df['category'].value_counts() / len(df) * 100).round(2))

Detected hierarchical structure: category/label/image.jpg

=== Summary by Label ===
label
neitherFireNorSmoke    52073
smoke                  29829
bothFireAndSmoke       27972
fire                   12760
Name: count, dtype: int64

=== Summary by Category ===
category
CV     95314
UAV    25097
RS      2223
Name: count, dtype: int64

=== Summary by Label and Category ===
category                CV    RS    UAV
label                                  
bothFireAndSmoke     20151     0   7821
fire                 12550     0    210
neitherFireNorSmoke  39199   888  11986
smoke                23414  1335   5080

=== Total Images: 122634 ===

=== Distribution Percentages ===

By Label:
label
neitherFireNorSmoke    42.46
smoke                  24.32
bothFireAndSmoke       22.81
fire                   10.40
Name: count, dtype: float64

By Category:
category
CV     77.72
UAV    20.46
RS      1.81
Name: count, dtype: float64
